# 03 — Table commune

**Étape du pipeline** : *Data transformation → Data tables*. C'est la table que D3 chargera avant le *visual mapping*.

Entrées : `data/interim/airbnb.parquet` et `data/interim/hotels.parquet`.
Sorties, dans `data/processed/` : `offres.csv` (table complète) et un `offres_<ville>.csv` par ville, pour la consigne « changer de dataset ».

In [ ]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prep").is_dir())
sys.path.insert(0, str(RACINE))
RAW, INTERIM, PROCESSED = (RACINE / "data" / d for d in ("raw", "interim", "processed"))

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

from src.prep import fusion, schema
from src.prep.common import PERCENTILES_ABERRANTS

## 1. Fusion

In [ ]:
sources = [schema.conformer(pd.read_parquet(INTERIM / f)) for f in ("airbnb.parquet", "hotels.parquet")]
toutes = fusion.fusionner(*sources)
print(toutes.shape)
toutes.groupby(["ville", "source"], observed=True).size().unstack()

## 2. Prix aberrants

On écarte les prix hors des percentiles 1 %–99 %, calculés **par ville et par source**.

In [ ]:
offres, ecartees = fusion.retirer_aberrants(toutes)
fusion.bilan_effectifs(toutes, offres)

In [ ]:
ecartees.groupby(["ville", "source"], observed=True)["prix_nuit_eur"].agg(["min", "max"]).round(0)

## 3. Contrôles qualité

### Valeurs manquantes par source (%)

In [ ]:
fusion.valeurs_manquantes(offres)

Les colonnes vides d'une source sont **attendues** : il n'y a pas d'étoiles côté Airbnb, ni de coordonnées côté hôtels, etc. Les colonnes obligatoires sont contrôlées par `schema.valider`.

### Prix et notes par ville et par source

In [ ]:
offres.groupby(["ville", "source"], observed=True).agg(
    n=("id", "size"),
    prix_median=("prix_nuit_eur", "median"),
    prix_q1=("prix_nuit_eur", lambda s: s.quantile(.25)),
    prix_q3=("prix_nuit_eur", lambda s: s.quantile(.75)),
    note_mediane=("note_100", "median"),
    metro=("metro_proche", "mean"),
).round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (v, g) in zip(axes, offres.groupby("ville", observed=True)):
    ax.boxplot([g.loc[g.source == s, "prix_nuit_eur"] for s in ("Airbnb", "Hôtel")], showfliers=False)
    ax.set_xticks([1, 2], ["Airbnb", "Hôtel"]); ax.set_title(v)
axes[0].set_ylabel("€ / nuit"); plt.tight_layout()

**Notes**

- Les notes Airbnb sont très concentrées vers le haut (médiane autour de 95/100), davantage que les notes Booking. Cette différence vient des systèmes de notation des deux plateformes, pas forcément de la qualité. C'est une limite à signaler dans le rapport.
- Les prix des hôtels de Rome restent hors d'échelle (`prix_comparable = False`, voir le notebook 02).

### Cohérence des distances au centre

In [ ]:
offres.groupby(["ville", "source"], observed=True)["dist_centre_km"].describe(percentiles=[.5]).round(1)

In [ ]:
pd.crosstab([offres.ville, offres.source], offres.tranche_dist, normalize="index").round(2)

Les deux sources ont des distributions de distance comparables : c'est ce qui permet de les comparer par tranche.

Attention : chaque dataset définit le « centre » à sa façon, et aucun des deux ne le documente.

### Rattachement des hôtels à un quartier

In [ ]:
h = offres[offres.source == "Hôtel"]
h.groupby("ville", observed=True)["quartier"].agg(taux_renseigne=lambda s: s.notna().mean(), nb_quartiers="nunique").round(3)

Ce taux mesure seulement si le **nom** du quartier est renseigné. Le taux de rattachement réel à un contour GeoJSON sera mesuré à l'étape « quartiers ».

## 4. Export pour D3

In [ ]:
chemins = fusion.exporter(offres, PROCESSED)
for p in chemins:
    print(f"{p.relative_to(RACINE)}  {p.stat().st_size / 1e6:.2f} Mo")

In [ ]:
# Relecture : ce que D3 verra avec d3.csv(..., d3.autoType)
pd.read_csv(PROCESSED / "offres_paris.csv").sample(5, random_state=0)